# 评测与真机反馈闭环

本 Demo 分别介绍固定基准评测、分级部署、遥测、失败归因和定向数据回流 API。


## 0. 初始化平台

只定义工作区与输出目录；模型、基准、机器人与方法参数紧邻 API 展示。


In [ ]:
# ruff: noqa
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("未找到 PhysReal-X 项目根目录。")
    PROJECT_ROOT = PROJECT_ROOT.parent
for import_root in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))
from examples._common import operation_receipt, write_summary
from physrealx import PhysRealX
from physrealx.errors import PhysRealXError

WORKSPACE = PROJECT_ROOT / ".physrealx/examples/06-evaluation-feedback"
OUTPUT_DIR = PROJECT_ROOT / "examples/evaluation_and_feedback/output"
px = PhysRealX.open(WORKSPACE)
receipts = {}

## 1. 执行固定基准：`evaluations.run`

**用途**：按冻结的任务、初始状态、种子、超时和成功谓词评测模型。

**输出**：逐 Episode 证据、成功率、95% 置信区间、安全事件和延迟。


In [ ]:
evaluation_model = None  # 替换为 ModelBundle/1 VersionedRef
benchmark_suite = None  # 替换为冻结 BenchmarkSuite/1 ArtifactRef
evaluation_operation = None
if evaluation_model is None or benchmark_suite is None:
    print("[BLOCKED] 请在本单元格绑定模型和固定基准。")
else:
    try:
        evaluation_operation = px.evaluations.run(
            evaluation_model,
            benchmark_suite,
            method="libero",
            config={"seeds": [0, 1, 2], "timeout_seconds": 300},
        )
        receipts["evaluation"] = operation_receipt(evaluation_operation)
        print(json.dumps(evaluation_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 2. 生成安全部署计划：`deployments.stage`

**用途**：将模型动作空间与 ROS2 机器人接口、控制周期和安全限制绑定。

**边界**：此 API 只发布计划；真机依次经过 shadow、限速/限力 canary 和 active。


In [ ]:
deployment_model = None  # 替换为通过基准的 ModelBundle/1 VersionedRef
robot = None  # 替换为 RobotInterfaceSpec/1 VersionedRef
deployment_operation = None
if deployment_model is None or robot is None:
    print("[BLOCKED] 请在本单元格绑定模型和机器人接口。")
else:
    try:
        deployment_operation = px.deployments.stage(
            deployment_model,
            robot,
            method="ros2_control",
            config={"stage": "shadow"},
        )
        receipts["deployment_plan"] = operation_receipt(deployment_operation)
        print(json.dumps(deployment_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 3. 记录真机遥测：`telemetry.record`

**用途**：对真机执行的观测、动作、TF、接触/力、诊断与任务谓词进行 MCAP 留证。

**输出**：可回放 `TelemetryBundle/1`，并绑定部署计划和模型版本。


In [ ]:
deployment_run = None  # 替换为通过安全门禁的 DeploymentRun/1 ArtifactRef
telemetry_operation = None
if deployment_run is None:
    print("[BLOCKED] 请在本单元格绑定真机部署运行。")
else:
    try:
        telemetry_operation = px.telemetry.record(
            deployment_run,
            method="rosbag2_mcap",
            config={"record_tf": True, "record_diagnostics": True},
        )
        receipts["telemetry"] = operation_receipt(telemetry_operation)
        print(json.dumps(telemetry_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 4. 失败归因：`failures.analyze`

**用途**：根据任务谓词、命令—状态误差、接触、TF、时钟和系统诊断确定开始偏离时刻。

**输出**：`cause_code`、`onset`、`confidence` 和证据引用；不确定时保留 `UNKNOWN`。


In [ ]:
telemetry_bundle = None  # 替换为已验证 TelemetryBundle/1 ArtifactRef
failure_operation = None
if telemetry_bundle is None:
    print("[BLOCKED] 请在本单元格绑定已验证遥测包。")
else:
    try:
        failure_operation = px.failures.analyze(
            telemetry_bundle,
            method="rules_v1",
            config={"retain_unknown": True},
        )
        receipts["failure_analysis"] = operation_receipt(failure_operation)
        print(json.dumps(failure_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 5. 构建定向反馈数据：`feedback.build_dataset`

**用途**：从人工确认的失败、OOD 和干预样本构建 hard-negative 或 DAgger 数据集。

**版本规则**：始终发布新数据集版本，禁止原地修改旧训练集。


In [ ]:
confirmed_failure_query = None  # 替换为人工确认 FailureQuery/1 ArtifactRef
feedback_operation = None
if confirmed_failure_query is None:
    print("[BLOCKED] 请在本单元格绑定已确认失败查询。")
else:
    try:
        feedback_operation = px.feedback.build_dataset(
            confirmed_failure_query,
            method="hard_negative",
            config={"new_version": "1.1.0"},
        )
        receipts["feedback_dataset"] = operation_receipt(feedback_operation)
        print(json.dumps(feedback_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 6. 保存回执

回执保存真实评测、部署、遥测、归因与数据回流记录。


In [ ]:
report = write_summary(
    OUTPUT_DIR / "evaluation-feedback-report.json",
    {"module": "评测与真机反馈闭环", "receipts": receipts},
)
print(json.dumps(report, ensure_ascii=False, indent=2, default=str))